In [ ]:
# 0) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 1) Cài đặt thư viện
!pip install -q segmentation-models-pytorch==0.3.4 albumentations==1.4.0
import torch, cv2, numpy as np, os, glob, random
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

# 2) Dataset
DATA_ROOT = '/content/drive/MyDrive/TJDR'
IMG_DIR  = os.path.join(DATA_ROOT, 'train/image')
MASK_DIR = os.path.join(DATA_ROOT, 'train/annotation')

img_files  = sorted(glob.glob(os.path.join(IMG_DIR,  '*')))
mask_files = sorted(glob.glob(os.path.join(MASK_DIR, '*')))

GRAY2CLS={0:0,14:1,38:2,75:3,113:4}
def load_mask(mp):
    raw=cv2.imread(mp,cv2.IMREAD_UNCHANGED)
    if raw.ndim==3: raw=cv2.cvtColor(raw,cv2.COLOR_BGR2GRAY)
    out=np.zeros(raw.shape,np.uint8)
    for g,c in GRAY2CLS.items(): out[raw==g]=c
    return out

def stem(p): return os.path.splitext(os.path.basename(p))[0]
mask_by_stem={stem(m):m for m in mask_files}
pairs=[(ip,mask_by_stem[stem(ip)]) for ip in img_files if stem(ip) in mask_by_stem]

SEED=42; random.seed(SEED); random.shuffle(pairs)
n=len(pairs); n_tr=int(0.7*n); n_va=int(0.15*n)
train_pairs=pairs[:n_tr]; val_pairs=pairs[n_tr:n_tr+n_va]; test_pairs=pairs[n_tr+n_va:]

# 3) Dataset + Augment
SIZE, NUM_CLASSES, BATCH = 768, 5, 4
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)

train_tf=A.Compose([
    A.Resize(SIZE,SIZE),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ElasticTransform(p=0.3),
    A.GridDistortion(p=0.3),
    A.CLAHE(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=MEAN,std=STD), ToTensorV2()
])
val_tf=A.Compose([A.Resize(SIZE,SIZE), A.Normalize(mean=MEAN,std=STD), ToTensorV2()])

class TJDRDataset(Dataset):
    def __init__(self,pairs,tf): self.pairs,self.tf=pairs,tf
    def __len__(self): return len(self.pairs)
    def __getitem__(self,i):
        ip,mp=self.pairs[i]
        img=cv2.cvtColor(cv2.imread(ip),cv2.COLOR_BGR2RGB)
        a=self.tf(image=img,mask=load_mask(mp))
        return a['image'],a['mask'].long()

train_dl=DataLoader(TJDRDataset(train_pairs,train_tf),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)
val_dl=DataLoader(TJDRDataset(val_pairs,val_tf),batch_size=BATCH,shuffle=False,num_workers=2,pin_memory=True)
test_dl=DataLoader(TJDRDataset(test_pairs,val_tf),batch_size=1,shuffle=False)

# 4) Model EfficientNet-B4 U-Net + Dropout
device='cuda' if torch.cuda.is_available() else 'cpu'
model=smp.Unet(encoder_name="efficientnet-b4",encoder_weights="imagenet",
               in_channels=3,classes=NUM_CLASSES,decoder_use_batchnorm=True,
               decoder_attention_type=None).to(device)

# 5) Loss + Optimizer + Scheduler
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
dice_loss=DiceLoss(mode='multiclass')
focal_loss=FocalLoss(mode='multiclass', alpha=0.25, gamma=2.0)
def criterion(logits,y): return dice_loss(logits,y)+focal_loss(logits,y)

opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=100)
scaler=torch.cuda.amp.GradScaler()

# 6) Evaluate Dice/IoU
LESION={1:'EX',2:'HE',3:'MA',4:'SE'}
@torch.no_grad()
def evaluate(dl):
    model.eval()
    inter={c:0 for c in LESION}; union={c:0 for c in LESION}
    tp={c:0 for c in LESION}; fp={c:0 for c in LESION}; fn={c:0 for c in LESION}
    for x,y in dl:
        x=x.to(device); y=y.to(device)
        pred=model(x).argmax(1)
        for c in LESION:
            p=(pred==c); t=(y==c)
            inter[c]+=(p&t).sum().item(); union[c]+=(p|t).sum().item()
            tp[c]+=(p&t).sum().item(); fp[c]+=(p&~t).sum().item(); fn[c]+=(~p&t).sum().item()
    rows={}
    for c in LESION:
        dice=2*tp[c]/(2*tp[c]+fp[c]+fn[c]) if (2*tp[c]+fp[c]+fn[c])>0 else float('nan')
        iou=inter[c]/union[c] if union[c]>0 else float('nan')
        rows[LESION[c]]=(dice,iou)
    return rows

# 7) Train loop với EarlyStopping + Checkpoint
import numpy as np
best=-1; patience=15; wait=0
EPOCHS=100
for ep in range(1,EPOCHS+1):
    model.train(); run=0
    for x,y in train_dl:
        x=x.to(device); y=y.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            logits=model(x); loss=criterion(logits,y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        run+=loss.item()
    scheduler.step()
    rows=evaluate(val_dl)
    mdice=np.nanmean([v[0] for v in rows.values()])
    print(f"Epoch {ep:02d} | loss {run/len(train_dl):.4f} | val mDice {mdice:.3f} | {rows}")
    if mdice>best:
        best=mdice; torch.save(model.state_dict(),'tjdr_unet_best.pth')
        print(" -> lưu model tốt nhất"); best=mdice; wait=0
    else:
        wait+=1
        if wait>=patience:
            print("Early stopping triggered"); break

# 8) Test
model.load_state_dict(torch.load('tjdr_unet_best.pth'))
rows=evaluate(test_dl)
print("KẾT QUẢ TEST:",rows)

# 9) Save model về Drive
import shutil
shutil.copy('tjdr_unet_best.pth','/content/drive/MyDrive/tjdr_unet_best.pth')
print("✅ Model đã lưu vào Drive")


Mounted at /content/drive
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.5/109.5 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 62.4 MB/s eta 0:00:00
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b4-6ed6700e.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b4-6ed6700e.pth


100%|██████████| 74.4M/74.4M [00:00<00:00, 88.9MB/s]
/tmp/ipykernel_2351/708789095.py:80: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler=torch.cuda.amp.GradScaler()


Epoch 01 | loss 1.1946 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0012106401820444126, 0.0006056867253659357), 'MA': (0.00022138587558113792, 0.00011070519207350825), 'SE': (0.00012763986192888038, 6.382400420798263e-05)}
 -> lưu model tốt nhất
Epoch 02 | loss 0.8162 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 03 | loss 0.6732 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 04 | loss 0.6281 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 05 | loss 0.5872 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 06 | loss 0.5744 | val mDice 0.000 | {'EX': (0.0, 0.0), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 07 | loss 0.5755 | val mDice 0.000 | {'EX': (nan, nan), 'HE': (0.0, 0.0), 'MA': (0.0, 0.0), 'SE': (0.0, 0.0)}
Epoch 08 | loss 0.5664 | val mDice 0.000 | {'EX': (nan